In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

In [3]:
df_sellers = spark.read.csv('olist_sellers_dataset.csv', header=True, inferSchema=True)
df_sellers.show()

+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
|c240c4061717ac180...|                 20920|   rio de janeiro|          RJ|
|e49c26c3edfa46d22...|                 55325|           brejao|          PE|
|1b938a7ec6ac5061a...|                 16304|        penapolis|          SP|
|768a86e36ad6aae3d...|                  1529|        sao paulo|          SP|
|ccc4bbb5f32a6ab2b...|                 80310|         curitiba|          PR|

In [4]:
from pyspark.sql.functions import col, upper, regexp_replace, when, split
from pyspark.sql.types import StringType

In [5]:
# Converter 'seller_zip_code_prefix' para String
df_sellers = df_sellers.withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast(StringType()))

# Filtrar por estado 'SP'
df_sellers_sp = df_sellers.filter(col("seller_state") == "SP")
df_sellers_sp.show()

+--------------------+----------------------+--------------------+------------+
|           seller_id|seller_zip_code_prefix|         seller_city|seller_state|
+--------------------+----------------------+--------------------+------------+
|3442f8959a84dea7e...|                 13023|            campinas|          SP|
|d1b65fc7debc3361e...|                 13844|          mogi guacu|          SP|
|c0f3eea2e14555b6f...|                  4195|           sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|   braganca paulista|          SP|
|1b938a7ec6ac5061a...|                 16304|           penapolis|          SP|
|768a86e36ad6aae3d...|                  1529|           sao paulo|          SP|
|a7a9b880c49781da6...|                 13530|           itirapina|          SP|
|8bd0f31cf0a614c65...|                  1222|           sao paulo|          SP|
|05a48cc8859962767...|                  5372|           sao paulo|          SP|
|f9ec7093df3a7b346...|                  

In [6]:
# Colocar nomes de cidade em letra maiúscula 
df_sellers_sp = df_sellers_sp.withColumn('seller_city', upper(col('seller_city')))

# Remover acentos
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), '[áàâãä]', 'a'))
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), '[éèêë]', 'e'))
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), '[íìîï]', 'i'))
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), '[óòôõö]', 'o'))
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), '[úùûü]', 'u'))
df_sellers_sp = df_sellers_sp.withColumn('seller_city', regexp_replace(col('seller_city'), 'ç', 'c'))

# Remover a coluna customer_state
df_sellers_sp = df_sellers_sp.drop("seller_state")

df_sellers_sp.show()

+--------------------+----------------------+--------------------+
|           seller_id|seller_zip_code_prefix|         seller_city|
+--------------------+----------------------+--------------------+
|3442f8959a84dea7e...|                 13023|            CAMPINAS|
|d1b65fc7debc3361e...|                 13844|          MOGI GUACU|
|c0f3eea2e14555b6f...|                  4195|           SAO PAULO|
|51a04a8a6bdcb23de...|                 12914|   BRAGANCA PAULISTA|
|1b938a7ec6ac5061a...|                 16304|           PENAPOLIS|
|768a86e36ad6aae3d...|                  1529|           SAO PAULO|
|a7a9b880c49781da6...|                 13530|           ITIRAPINA|
|8bd0f31cf0a614c65...|                  1222|           SAO PAULO|
|05a48cc8859962767...|                  5372|           SAO PAULO|
|f9ec7093df3a7b346...|                  5138|           SAO PAULO|
|4e6015589b781adaa...|                 11440|             GUARUJA|
|4cf490a58259286ad...|                 14910|           TABATI

## Detectando e Corrigindo Nomes de Cidades
Para detectar e corrigir a cidade "jacarei / sao paulo" para apenas "jacarei", podemos usar a função `when` para aplicar uma condição. Se a coluna `seller_city` contiver "/", pegamos a parte antes da barra, caso contrário, mantemos o nome original. Isso é eficiente mesmo para milhões de linhas.

In [7]:

df_sellers_sp = df_sellers_sp.withColumn('seller_city', 
    when(col('seller_city').contains('/'), split(col('seller_city'), ' / ').getItem(0))
    .otherwise(col('seller_city'))
)

df_sellers_sp.show()


+--------------------+----------------------+--------------------+
|           seller_id|seller_zip_code_prefix|         seller_city|
+--------------------+----------------------+--------------------+
|3442f8959a84dea7e...|                 13023|            CAMPINAS|
|d1b65fc7debc3361e...|                 13844|          MOGI GUACU|
|c0f3eea2e14555b6f...|                  4195|           SAO PAULO|
|51a04a8a6bdcb23de...|                 12914|   BRAGANCA PAULISTA|
|1b938a7ec6ac5061a...|                 16304|           PENAPOLIS|
|768a86e36ad6aae3d...|                  1529|           SAO PAULO|
|a7a9b880c49781da6...|                 13530|           ITIRAPINA|
|8bd0f31cf0a614c65...|                  1222|           SAO PAULO|
|05a48cc8859962767...|                  5372|           SAO PAULO|
|f9ec7093df3a7b346...|                  5138|           SAO PAULO|
|4e6015589b781adaa...|                 11440|             GUARUJA|
|4cf490a58259286ad...|                 14910|           TABATI

In [8]:
# Salvar o DataFrame agregado em CSV usando Pandas (evita dependencia do Hadoop no Windows)
import csv
import os
import pandas as pd
from datetime import datetime

df_sellers_final_df = df_sellers_sp.toPandas()
output_dir = os.getcwd()
output_path = os.path.join(output_dir, f"sellers_final_{datetime.now():%Y%m%d_%H%M%S}.csv")
df_sellers_final_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

CSV salvo em: c:\Users\Sofhia\Downloads\archive\sellers_final_20260331_232908.csv
